# Phase 5 — FastAPI Backend

**Goal:** Wrap the RAG pipeline from Phase 4 into a clean HTTP API with three endpoints:

- `POST /upload`   — upload a PDF, parse and index it.
- `POST /ask`      — ask a question; return answer + citations.
- `GET  /documents` — list indexed documents.

Plus `/health` for readiness probes and `/docs` for the auto-generated OpenAPI UI.

This is where the project starts looking production-shaped. We use Pydantic v2 for typed request/response models, dependency injection for shared resources, and `httpx` to call the live server from this notebook.


## 5.1 — Why FastAPI?

- Type-driven (Pydantic v2) — request/response validation is free.
- Async-friendly — useful for streaming LLM responses later.
- Auto OpenAPI + Swagger UI — every endpoint gets a doc page at `/docs`.
- Production-grade ASGI server (`uvicorn`) — supports workers, hot-reload, graceful shutdown.

Alternatives considered: Flask (no native async or types), Starlette (lower level), Litestar (newer, smaller ecosystem).


## 5.2 — Pydantic request / response schemas


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class AskRequest(BaseModel):
    question: str = Field(..., min_length=3, max_length=2000)
    k: int = Field(5, ge=1, le=20)
    doc_id: Optional[str] = Field(None, description="If set, only search within this document.")

class RetrievedChunk(BaseModel):
    page: int
    score: float
    chunk_id: str
    preview: str

class AskResponse(BaseModel):
    question: str
    answer: str
    cited_pages: list[int]
    retrieved: list[RetrievedChunk]
    latency_s: float
    model: str

class UploadResponse(BaseModel):
    doc_id: str
    source: str
    n_pages: int
    n_chunks: int

class DocumentInfo(BaseModel):
    doc_id: str
    source: str
    n_pages: int
    n_chunks: int


## 5.3 — The FastAPI app

Here is the file that lives at `apps/api/main.py`. We use FastAPI's lifespan event to load the embedder, vector store, and LLM exactly once at startup.


In [ ]:
api_src = r'''
"""apps/api/main.py — production entry point."""
from contextlib import asynccontextmanager
from pathlib import Path

from fastapi import Depends, FastAPI, File, HTTPException, UploadFile
from sentence_transformers import SentenceTransformer

from mrta.core.config import settings
from apps.api.schemas import AskRequest, AskResponse, UploadResponse, DocumentInfo, RetrievedChunk
from mrta.core.rag_pipeline import build_pipeline, RagPipeline
from mrta.ingestion.pdf_loader import load_pdf
from mrta.ingestion.chunker import token_chunks
from mrta.retrieval.vector_store import VectorStore

state: dict = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    embedder = SentenceTransformer(settings.embedding_model)
    store_dir = settings.vector_store_path / "default"
    if (store_dir / "index.faiss").exists():
        store = VectorStore.load(store_dir, embedder)
    else:
        store = VectorStore(dim=embedder.get_sentence_embedding_dimension(), embedder=embedder)
    state["pipeline"] = build_pipeline(store=store)
    state["store"] = store
    state["embedder"] = embedder
    yield
    state.clear()

app = FastAPI(title="Multimodal RAG Assistant", version="0.1.0", lifespan=lifespan)

def get_pipeline() -> RagPipeline:
    return state["pipeline"]

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/upload", response_model=UploadResponse)
async def upload(file: UploadFile = File(...)):
    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(400, "Only PDF uploads are supported.")
    raw_dir = Path("data/raw"); raw_dir.mkdir(parents=True, exist_ok=True)
    path = raw_dir / file.filename
    path.write_bytes(await file.read())
    pdf = load_pdf(path)
    chunks = token_chunks(pdf, size=settings.chunk_size, overlap=settings.chunk_overlap)
    state["store"].add(chunks)
    state["store"].save(settings.vector_store_path / "default")
    return UploadResponse(doc_id=pdf.doc_id, source=pdf.source, n_pages=pdf.n_pages, n_chunks=len(chunks))

@app.post("/ask", response_model=AskResponse)
def ask(req: AskRequest, pipe: RagPipeline = Depends(get_pipeline)):
    return AskResponse(**pipe.run(req.question, k=req.k, doc_id=req.doc_id))

@app.get("/documents", response_model=list[DocumentInfo])
def documents():
    # Aggregate by doc_id from the vector store metadata.
    by_doc: dict[str, dict] = {}
    for m in state["store"].metadata:
        d = by_doc.setdefault(m["doc_id"], {"doc_id": m["doc_id"], "source": m["source"], "pages": set(), "n_chunks": 0})
        d["pages"].add(m["page"]); d["n_chunks"] += 1
    return [DocumentInfo(doc_id=d["doc_id"], source=d["source"], n_pages=len(d["pages"]), n_chunks=d["n_chunks"]) for d in by_doc.values()]
'''
print(api_src[:1200])


## 5.4 — Running it locally

```bash
# From the repo root, with the venv activated:
uvicorn apps.api.main:app --reload --port 8000
```

Then visit:

- http://localhost:8000/docs — Swagger UI
- http://localhost:8000/health — should return `{"status": "ok"}`


## 5.5 — Driving the API from this notebook

Once the server is running in another terminal, the rest of this notebook talks to it over HTTP. This is exactly how the Streamlit UI in Phase 6 will talk to the backend, so it is good practice.


In [ ]:
import httpx

BASE = "http://localhost:8000"

def alive() -> bool:
    try:
        r = httpx.get(f"{BASE}/health", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

if alive():
    print("API is up.")
else:
    print("API is not running. Start it in another terminal:")
    print("  uvicorn app.api.main:app --reload --port 8000")


## 5.6 — Upload a PDF


In [ ]:
from pathlib import Path

pdf_path = Path("data/raw/attention_is_all_you_need.pdf")
if not pdf_path.exists():
    raise FileNotFoundError("Run Notebook 01 first to download the sample PDF.")

if alive():
    with pdf_path.open("rb") as f:
        files = {"file": (pdf_path.name, f, "application/pdf")}
        r = httpx.post(f"{BASE}/upload", files=files, timeout=120)
    r.raise_for_status()
    print(r.json())
else:
    print("Skipping — server not running.")


## 5.7 — Ask a question


In [ ]:
if alive():
    payload = {"question": "What is multi-head attention and why is it useful?", "k": 5}
    r = httpx.post(f"{BASE}/ask", json=payload, timeout=120)
    r.raise_for_status()
    resp = r.json()
    print("Q:", resp["question"])
    print("\nA:", resp["answer"])
    print("\ncited:", resp["cited_pages"], " | latency:", round(resp["latency_s"], 2), "s")


## 5.8 — List documents


In [ ]:
if alive():
    r = httpx.get(f"{BASE}/documents")
    r.raise_for_status()
    for d in r.json():
        print(f"  {d['doc_id']:30s}  {d['source']:40s}  pages={d['n_pages']}  chunks={d['n_chunks']}")


## 5.9 — Production-shaped touches

Things to add when you graduate from "works on my machine":

| Concern         | Recommended addition                                                            |
|-----------------|----------------------------------------------------------------------------------|
| Auth            | `fastapi.security.APIKeyHeader` + a `Depends(verify_key)` on write endpoints.    |
| Rate limiting   | `slowapi` middleware, per-IP buckets.                                            |
| Tracing         | `opentelemetry-instrumentation-fastapi`; export to Jaeger or LangSmith.          |
| Errors          | Global exception handler that returns JSON, not HTML.                            |
| Validation      | Use `pydantic.Field(...)` ranges (we did); reject empty PDFs at upload.          |
| Streaming       | Switch `/ask` to `StreamingResponse` and stream the LLM tokens.                  |
| Background jobs | For large PDFs, return `202 Accepted` + a job_id; index in a Celery worker.      |

We will add Docker + logging in Notebook 09. Streaming is a great post-tutorial exercise.


## What you learned

- The full set of endpoints needed for a RAG app: `upload`, `ask`, `documents`, `health`.
- How to use Pydantic v2 for request/response validation.
- The lifespan pattern for loading expensive resources once.
- Driving the API from Python via `httpx`.
- A roadmap from "works on my laptop" to production-shaped.

## Exercises

1. Add a `DELETE /documents/{doc_id}` endpoint that removes a doc + its chunks (needs a vector store that supports filtering, e.g. Qdrant).
2. Convert `/ask` to a streaming endpoint that yields tokens as the LLM produces them.
3. Add a `/ask` rate limit of 30 requests/minute per IP.

## Next: [Phase 6 — Streamlit frontend](./2026-05-25-phase06-streamlit-frontend.ipynb)
